# Episode 5 — Pytrees

**Instructor notebook** · run top-to-bottom before recording.

JAX treats nested **dict / list / tuple** structures of arrays as **pytrees**. Model parameters, batches, and optimizer state all use this pattern — and utilities like **`jax.tree.map`** let you transform every leaf at once.

| | |
|---|---|
| **Chapter** | 1.5 · Part I — Pure JAX |
| **Prereq** | Episodes 1–4 |
| **Next** | Part II — GPT-2 transformer |

**Source:** [Pytrees](https://docs.jax.dev/en/latest/pytrees.html) · [`jax.tree.map`](https://docs.jax.dev/en/latest/_autosummary/jax.tree.map.html) · [`jax.tree.leaves`](https://docs.jax.dev/en/latest/_autosummary/jax.tree.leaves.html)


In [1]:
import collections

import jax
import jax.numpy as jnp
import numpy as np


## What is a pytree?

A pytree is built from **nodes** (`list`, `tuple`, `dict`) and **leaves** (arrays, scalars, or anything not registered as a container). JAX walks the nodes and collects the leaves — that's how `grad`, `jit`, and `vmap` handle whole parameter trees.

In ML you'll see pytrees everywhere: model weights, dataset rows, RL observations. `jax.tree.leaves` is the quick way to count or inspect them.


In [2]:
example_trees = [
    [1, 'a', object()],
    (1, (2, 3), ()),
    [1, {'k1': 2, 'k2': (3, 4)}, 5],
    {'a': 2, 'b': (2, 3)},
    jnp.array([1, 2, 3]),
]

for pytree in example_trees:
    leaves = jax.tree.leaves(pytree)
    print(f"{repr(pytree):<45} has {len(leaves)} leaves: {leaves}")


[1, 'a', <object object at 0xf0e8348e0a00>]   has 3 leaves: [1, 'a', <object object at 0xf0e8348e0a00>]
(1, (2, 3), ())                               has 3 leaves: [1, 2, 3]
[1, {'k1': 2, 'k2': (3, 4)}, 5]               has 5 leaves: [1, 2, 3, 4, 5]
{'a': 2, 'b': (2, 3)}                         has 3 leaves: [2, 2, 3]
Array([1, 2, 3], dtype=int32)                 has 1 leaves: [Array([1, 2, 3], dtype=int32)]


Lists, tuples, and dicts are in JAX's built-in registry. Custom classes can be registered too — see [Custom pytree nodes](https://docs.jax.dev/en/latest/custom_pytrees.html) when you outgrow plain dicts.


### `NamedTuple` is a pytree

`collections.namedtuple` (and `typing.NamedTuple`) register as container nodes — fields are flattened by **attribute name**, same as in the key-path examples later.

In [4]:
from typing import NamedTuple

class Layer(NamedTuple):
    w: jnp.ndarray
    b: jnp.ndarray

layer = Layer(w=jnp.ones((2, 3)), b=jnp.zeros(3))
print("leaves:", jax.tree.leaves(layer))
print("tree.map:", jax.tree.map(lambda x: x * 2, layer))
print("tree.map.shapes:", jax.tree.map(lambda x: x.shape, layer))

leaves: [Array([[1., 1., 1.],
       [1., 1., 1.]], dtype=float32), Array([0., 0., 0.], dtype=float32)]
tree.map: Layer(w=Array([[2., 2., 2.],
       [2., 2., 2.]], dtype=float32), b=Array([0., 0., 0.], dtype=float32))
tree.map.shapes: Layer(w=(2, 3), b=(3,))


## `jax.tree.map`

The workhorse pytree utility. Like Python's `map`, but applied leaf-by-leaf across an entire tree. Pass two or more trees and the function receives matching leaves from each — **structures must match** (same keys, same list lengths).


In [6]:
list_of_lists = [
    [1, 2, 3],
    [1, 2],
    [1, 2, 3, 4],
]

jax.tree.map(lambda x: x * 2, list_of_lists)


[[2, 4, 6], [2, 4], [2, 4, 6, 8]]

In [8]:
jax.tree.map(lambda x, y: x + y, list_of_lists, list_of_lists)


[[2, 4, 6], [2, 4], [2, 4, 6, 8]]

## MLP params + SGD update

A common pattern: store params as a **list of layer dicts**, inspect shapes with `tree.map`, then update with SGD. `jax.grad` returns a pytree with the **same structure** as `params` — one `tree.map` subtracts the learning-rate-scaled gradient from every weight and bias.


In [ ]:
def init_mlp_params(layer_widths):
    params = []
    for n_in, n_out in zip(layer_widths[:-1], layer_widths[1:]):
        params.append(dict(
            weights=np.random.normal(size=(n_in, n_out)) * np.sqrt(2 / n_in),
            biases=np.ones(n_out),
        ))
    return params # List of PyTree

params = init_mlp_params([1, 128, 128, 1])


In [ ]:
jax.tree.map(lambda x: x.shape, params)


In [ ]:
def forward(params, x):
    *hidden, last = params
    for layer in hidden:
        x = jax.nn.relu(x @ layer['weights'] + layer['biases'])
    return x @ last['weights'] + last['biases']


def loss_fn(params, x, y):
    return jnp.mean((forward(params, x) - y) ** 2)


LEARNING_RATE = 0.0001


@jax.jit
def update(params, x, y):
    grads = jax.grad(loss_fn)(params, x, y)
    return jax.tree.map(lambda p, g: p - LEARNING_RATE * g, params, grads)


x_batch = jnp.ones((10, 1))
y_batch = x_batch ** 2
for i in range(10):
    loss, grads = jax.value_and_grad(loss_fn)(params, x_batch, y_batch)
    params = jax.tree.map(lambda p, g: p - LEARNING_RATE * g, params, grads)
    print(f"loss: {loss} at iteration {i}")





## `tree_structure`

When debugging, `tree_structure` shows how JAX classifies an object — useful when something you expect to be a leaf is actually treated as a node (or vice versa).


In [ ]:
from jax.tree_util import tree_structure

print(tree_structure(object))


## Pytrees and transforms

Transformations like `vmap` and `scan` accept pytrees of arrays. Their axis arguments (`in_axes`, `out_axes`) can be pytrees too, aligned with the argument structure.

You can also pass a **prefix** — a single value broadcast to an entire subtree:

```python
vmap(f, in_axes=(None, {"k1": None, "k2": 0}))  # only map over k2
vmap(f, in_axes=(None, 0))                         # map over whole dict
vmap(f, in_axes=0)                                 # map everything (default)
```


## Key paths

Every leaf has a unique **key path** — the sequence of indices/keys from root to leaf. Handy for debugging which parameter is which:

- `tree_flatten_with_path` — flatten with paths
- `tree_map_with_path` — map with paths passed to the function
- `keystr` — pretty-print a path like `['layer']['weights']`


In [ ]:
ATuple = collections.namedtuple("ATuple", ("name",))

tree = [1, {"k1": 2, "k2": (3, 4)}, ATuple("foo")]
flattened, _ = jax.tree_util.tree_flatten_with_path(tree)

for key_path, value in flattened:
    print(f"tree{jax.tree_util.keystr(key_path)} = {value}")


In [ ]:
for key_path, _ in flattened:
    print(f"tree{jax.tree_util.keystr(key_path)}: {repr(key_path)}")


## Gotchas

### Shapes are tuples (nodes), not leaves

An array's `.shape` is a Python tuple — a pytree **node**. Mapping `jnp.ones` over shapes calls it on each dimension separately, not on the pair `(2, 3)`. Fix: avoid the intermediate map, or wrap the shape in `jnp.array` to make it a leaf.


In [ ]:
a_tree = [jnp.zeros((2, 3)), jnp.zeros((3, 4))]
shapes = jax.tree.map(lambda x: x.shape, a_tree)
jax.tree.map(jnp.ones, shapes)  # maps over 2 and 3, not (2, 3)!


### `None` is absent, not a leaf

By default, `None` entries are **skipped** — not counted as leaves. Pass `is_leaf=lambda x: x is None` when you need to preserve them.


In [ ]:
jax.tree.leaves([None, None, None])
jax.tree.leaves([None, None, None], is_leaf=lambda x: x is None)


### Dict keys must be sortable

JAX flattens dicts by **sorted keys** (for stable structure / JIT caching). Mixing `int` and `str` keys that can't be ordered raises an error. Use `OrderedDict` or a custom pytree node if you need a different key order.


In [ ]:
try:
    jax.tree.map(lambda x: x + 1, {1: 7, "y": 42})
except ValueError as e:
    print(type(e).__name__ + ":", e)


## Transpose: list of trees → tree of lists

Common when batching dataset steps: turn `[{t: 1, obs: 3}, {t: 2, obs: 4}]` into `{t: [1, 2], obs: [3, 4]}`.

**Option 1:** `tree.map` with a variadic lambda (simple). **Option 2:** `tree.transpose` when you need explicit inner/outer structure control.


In [ ]:
def tree_transpose(list_of_trees):
    return jax.tree.map(lambda *xs: list(xs), *list_of_trees)


episode_steps = [dict(t=1, obs=3), dict(t=2, obs=4)]
tree_transpose(episode_steps)


In [ ]:
jax.tree.transpose(
    outer_treedef=jax.tree.structure([0 for _ in episode_steps]),
    inner_treedef=jax.tree.structure(episode_steps[0]),
    pytree_to_transpose=episode_steps,
)


For custom containers (Flax `Module`, dataclasses, etc.), register flatten/unflatten handlers — see [Custom pytree nodes](https://docs.jax.dev/en/latest/custom_pytrees.html).

---

**Next:** Part II — GPT-2 transformer.
